## 02 · 为什么需要RAG

本课程使用亚马逊美国站服饰箱包知识库，逐步做出一个能回答产品、法规和尺码问题的问答程序。

**这一课只看清问题：模型没有资料时会猜；把资料放进提示词后，答案才有依据。**

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
api_key = os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY")
base_url = os.getenv("LLM_BASE_URL")
LLM_MODEL = os.getenv("LLM_MODEL")
if api_key and not LLM_MODEL:
    raise RuntimeError("请在 .env 中配置 LLM_MODEL。")
client = OpenAI(api_key=api_key, base_url=base_url or None) if api_key else None
print(f"API 已配置, 模型使用{LLM_MODEL}" if client else "未配置 API：保留本地步骤，调用模型的单元会跳过")

API 已配置, 模型使用deepseek-v4-flash-0731


## 先问模型：它并不知道我们的内部资料

In [2]:
question = "SKU-YG301 瑜伽裤的面料成分是什么？"

if client:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=0,
    )
    print(response.choices[0].message.content)
else:
    print(question)

SKU-YG301 瑜伽裤的面料成分是什么？


## 手动把资料放进提示词

这次先不做自动检索，直接读取一份产品规格。后面的 Notebook 会把“找到哪段资料”也自动完成。

In [3]:
from pathlib import Path

product_file = Path("../data/产品/瑜伽裤-YG301/产品规格.md")
product_text = product_file.read_text(encoding="utf-8")
prompt = f"""请只根据资料回答问题；资料没有答案时请明确说不知道。

资料：
{product_text}

问题：{question}"""

if client:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    print(response.choices[0].message.content)
else:
    print(prompt[:1200])

请只根据资料回答问题；资料没有答案时请明确说不知道。

资料：
# 瑜伽裤 SKU-YG301 技术规格书

> 文档编号: DOC-PROD-YG301-TECH
> 商品代号: SKU-YG301 (Naked-Feel High-Waist Yoga Leggings)
> 类目: 女士运动瑜伽服
> 性别: 女款
> 季节: 2026 春夏
> 状态: Active

---

## 1. 产品概述

裸感无痕高腰瑜伽裤，定位北美中高端女性运动市场。目标用户为25-40岁瑜伽、普拉提、健身女性，追求裸感穿戴体验与深蹲防透光。

核心卖点：75% Nylon 66 + 25% Lycra四面弹深蹲零透光；230 GSM双面微磨毛Butter-soft触感；四针六线拼缝无摩擦；加宽高腰收腹隐藏钥匙袋。

![产品主图](../../images/Apparel/28456.jpg)

![面料细节](../../images/Apparel/26994.jpg)

---

## 2. 面料规格

### 2.1 纤维成分

75% Nylon 66 (锦纶/超细聚酰胺) + 25% Lycra Spandex (莱卡四面弹氨纶)。纱线40D/48F双面精密经编，克重230 GSM (±5g)。

Nylon 66熔点265°C(高于Nylon 6的220°C)，密度1.14 g/cm³，回潮率4.5%，吸湿快干。40D/48F: 40旦尼尔粗细，48根长丝束，单丝0.83旦尼尔超细柔软。

### 2.2 面料结构

经编(Warp-knitted)双面布：面层平纹致密组织提供防透光基础；底层微毛圈组织经碳素磨毛处理提供Butter-soft触感。克重230 GSM，厚度0.45mm，透气率85 mm/s。

### 2.3 功能性能

| 指标 | 测试方法 | 标准 | 实测 |
| :--- | :--- | :--- | :--- |
| 防透光 | SGS Squat-Proof | 透光率≤5% | ≤2% (5级) |
| 弹性恢复 | 500次循环拉伸 | 恢复率≥90% | ≥95% |
| 抗起球 | ASTM D3512 | ≥3.5级 | 4.5级 |
| 透气率 | ISO 9237 | ≥50mm/s | 85mm/s |
| 吸湿

## RAG 的三步

1. **Retrieve**：从知识库找到相关片段
2. **Augment**：把问题和片段放进提示词
3. **Generate**：让模型根据资料组织答案

接下来按这个顺序，一步一步写出来。